## Part 1: GDP by Country Stacked Bar Plot
In this section, I scrape the nominal GDP by country table from the Wikipedia page provided in the assignment. I use the IMF 2026 GDP estimates from the table, clean the country names and GDP values, assign countries to broad world regions, and create an interactive stacked bar plot using Plotly. Each bar represents a region, and countries are stacked within their corresponding region.

In [ ]:
import pandas as pd
import plotly.express as px

## Read GDP data from Wikipedia

The GDP data are scraped directly from the Wikipedia page using `pandas.read_html()`. Because Wikipedia block my automated requests at first, I include a browser-style user agent header with assistance from colab built-in AI. After reading all tables from the page, I inspect the first few tables and identify the table containing country-level GDP estimates from the IMF.

In [ ]:
url = "https://en.wikipedia.org/wiki/List_of_countries_by_GDP_(nominal)"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
}
tables = pd.read_html(url, storage_options=headers)
len(tables)

8

In [ ]:
for i, table in enumerate(tables[:5]):
    print("Table", i)
    print(table.head())
    print()

Table 0
                                                   0
0  Largest economies in the world by nominal GDP ...

Table 1
                                                   0  \
0  > $20 trillion $10–20 trillion $5–10 trillion ...   

                                                   1  
0  $250–500 billion $100–250 billion $50–100 bill...  

Table 2
  Country/Territory IMF (2026)[1] World Bank (2024)[6]  \
0             World     126295331            111326370   
1     United States      32383920             28750956   
2        China[n 1]      20851593             18743803   
3           Germany       5452858              4685593   
4             Japan       4379253              4027598   

  United Nations (2024)[7]  
0                100834796  
1                 29298000  
2                 18743802  
3                  4659929  
4                  4026211  

Table 3
                              Regional groupings  IMF (2026)[1]  \
0                                          Wor

## Select the IMF country GDP table
The country-level GDP table is stored as Table 2. I use the IMF estimate column because the assignment asks for the IMF numbers.

In [ ]:
gdp = tables[2]
gdp.head()

,Country/Territory,IMF (2026)[1],World Bank (2024)[6],United Nations (2024)[7]
0,World,126295331,111326370,100834796
1,United States,32383920,28750956,29298000
2,China[n 1],20851593,18743803,18743802
3,Germany,5452858,4685593,4659929
4,Japan,4379253,4027598,4026211


In [ ]:
gdp.columns

Index(['Country/Territory', 'IMF (2026)[1]', 'World Bank (2024)[6]',
       'United Nations (2024)[7]'],
      dtype='object')

## Clean country and GDP variables
I keep only the country name and IMF GDP estimate columns. The GDP values are converted from text to numeric values so that they can be used correctly as the y-axis values in the Plotly bar chart.

In [ ]:
gdp_clean = gdp.rename(columns={"Country/Territory": "country", "IMF (2026)[1]": "gdp"})
gdp_clean = gdp_clean[["country", "gdp"]]
gdp_clean.head()

,country,gdp
0,World,126295331
1,United States,32383920
2,China[n 1],20851593
3,Germany,5452858
4,Japan,4379253


In [ ]:
gdp_clean["gdp"] = (gdp_clean["gdp"].astype(str).str.replace(",", "", regex=False))
gdp_clean["gdp"] = pd.to_numeric(gdp_clean["gdp"],errors="coerce")
gdp_clean = gdp_clean.dropna()
gdp_clean.head()

,country,gdp
0,World,126295331.0
1,United States,32383920.0
2,China[n 1],20851593.0
3,Germany,5452858.0
4,Japan,4379253.0


## Assign countries to regions
To stack countries within broad regions, I merge the GDP table with Plotly's built-in Gapminder country-region information. Countries without a matched region are removed before plotting.

In [ ]:
gapminder = px.data.gapminder()
country_region = (gapminder[["country", "continent"]].drop_duplicates().rename(columns={"continent": "region"}))
gdp_region = gdp_clean.merge(country_region, on="country", how="left")
gdp_region.head()

,country,gdp,region
0,World,126295331.0,NaN
1,United States,32383920.0,Americas
2,China[n 1],20851593.0,NaN
3,Germany,5452858.0,Europe
4,Japan,4379253.0,Asia


In [ ]:
gdp_region = gdp_region.dropna(subset=["region"])

## Create the stacked interactive bar plot
The final figure is an interactive stacked bar plot. Each bar shows one broad region, and the colored segments within each bar represent individual countries and their IMF GDP estimates.

In [ ]:
fig_bar = px.bar(gdp_region, x="region", y="gdp", color="country", title="Nominal GDP by Country within Region Using IMF Estimates")
fig_bar.update_layout(
    barmode="stack",
    height=700
)
fig_bar.show()

## Save the stacked bar plot
I save the interactive Plotly figure as `stacked_bar.html`.

In [ ]:
fig_bar.write_html("stacked_bar.html")

## Part 2: MRICloud Sankey Diagram
In this section, I follow the interactive graphics lecture example that displays one subject's MRICloud data as a sunburst plot. I use the same Type 1 Level 5 MRICloud data and the same hierarchy lookup table. The Sankey diagram starts from intracranial volume (`ICV`) and shows the flow through multiple MRICloud hierarchy levels: `level1`, `level2`, and `level3`.

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go

### Load the MRICloud hierarchy lookup table
The lookup table provides the hierarchy for each MRICloud region. This is the same table used in the lecture example. I rename the columns so that they match the hierarchy levels used in the sunburst plot example.

In [ ]:
## load in the hierarchy information
url = "https://raw.githubusercontent.com/bcaffo/MRIcloudT1volumetrics/master/inst/extdata/multilevel_lookup_table.txt"
multilevel_lookup = pd.read_csv(url, sep="\t").drop(["Level5"], axis=1)
multilevel_lookup = multilevel_lookup.rename(columns={
    "modify": "roi",
    "modify.1": "level4",
    "modify.2": "level3",
    "modify.3": "level2",
    "modify.4": "level1"
})

multilevel_lookup = multilevel_lookup[["roi", "level4", "level3", "level2", "level1"]]
multilevel_lookup.head()

,roi,level4,level3,level2,level1
0,SFG_L,SFG_L,Frontal_L,CerebralCortex_L,Telencephalon_L
1,SFG_R,SFG_R,Frontal_R,CerebralCortex_R,Telencephalon_R
2,SFG_PFC_L,SFG_L,Frontal_L,CerebralCortex_L,Telencephalon_L
3,SFG_PFC_R,SFG_R,Frontal_R,CerebralCortex_R,Telencephalon_R
4,SFG_pole_L,SFG_L,Frontal_L,CerebralCortex_L,Telencephalon_L


### Load one subject's Type 1 Level 5 data
Following the lecture example, I use subject `127` and filter the MRICloud data to Type 1 and Level 5.

In [ ]:
# Load subject data
subject_id = 127
subjectData = pd.read_csv("https://raw.githubusercontent.com/smart-stats/ds4bio_book/main/book/assetts/kirby21AllLevels.csv")
subjectData = subjectData.loc[(subjectData.type == 1) & (subjectData.level == 5) & (subjectData.id == subject_id)]
subjectData = subjectData[["roi", "volume"]]
subjectData.head()

,roi,volume
217,SFG_L,12926
218,SFG_R,10050
219,SFG_PFC_L,12783
220,SFG_PFC_R,11507
221,SFG_pole_L,3078


### Merge subject data with hierarchy information
The subject data only contains ROI names and volume values. I merge it with the hierarchy lookup table so that each ROI has its corresponding hierarchy labels. For the Sankey diagram, I display the hierarchy from `ICV` through `level1`, `level2`, and `level3`.

In [ ]:
subjectData = pd.merge(subjectData, multilevel_lookup, on="roi")
subjectData = subjectData.assign(icv="ICV")
subjectData = subjectData.assign(comp=subjectData.volume / np.sum(subjectData.volume))
subjectData.head()

,roi,volume,level4,level3,level2,level1,icv,comp
0,SFG_L,12926,SFG_L,Frontal_L,CerebralCortex_L,Telencephalon_L,ICV,0.009350
1,SFG_R,10050,SFG_R,Frontal_R,CerebralCortex_R,Telencephalon_R,ICV,0.007270
2,SFG_PFC_L,12783,SFG_L,Frontal_L,CerebralCortex_L,Telencephalon_L,ICV,0.009247
3,SFG_PFC_R,11507,SFG_R,Frontal_R,CerebralCortex_R,Telencephalon_R,ICV,0.008324
4,SFG_pole_L,3078,SFG_L,Frontal_L,CerebralCortex_L,Telencephalon_L,ICV,0.002227


### Convert the hierarchy into Sankey links
Here, I convert the MRICloud hierarchy into links from one level to the next: `ICV → level1 → level2 → level3`. The `comp` variable is used as the flow value, so the links represent the relative composition of each region.

In [ ]:
def make_links(data, source_col, target_col, value_col):
    links = (data.groupby([source_col, target_col], as_index=False)[value_col].sum())
    links = links.rename(columns={source_col: "source", target_col: "target", value_col: "value"})
    return links

In [ ]:
sankey_links = pd.concat([
    make_links(subjectData, "icv", "level1", "comp"),
    make_links(subjectData, "level1", "level2", "comp"),
    make_links(subjectData, "level2", "level3", "comp")
], ignore_index=True)

sankey_links.head()

,source,target,value
0,ICV,CSF,0.079417
1,ICV,Diencephalon_L,0.008548
2,ICV,Diencephalon_R,0.008362
3,ICV,Mesencephalon,0.007430
4,ICV,Metencephalon,0.115313


### Create the Sankey diagram
Plotly Sankey diagrams require source and target values to be numeric indices rather than text labels. I first create a list of all unique node labels and then map each source and target name to its numeric index.

In [ ]:
labels = pd.unique(sankey_links[["source", "target"]].values.ravel())
label_to_index = {label: i for i, label in enumerate(labels)}
source = sankey_links["source"].map(label_to_index)
target = sankey_links["target"].map(label_to_index)
value = sankey_links["value"]

In [ ]:
fig_sankey = go.Figure(data=[go.Sankey(
    node=dict(
        pad=20,
        thickness=15,
        line=dict(color="black", width=0.5),
        label=labels
    ),
    link=dict(
        source=source,
        target=target,
        value=value
    )
)])
fig_sankey.update_layout(
    title_text="MRICloud Type 1 Subject Data as a Sankey Diagram",
    font_size=10,
    width=1800,
    height=1500
)
fig_sankey.show()

### Save the Sankey diagram as an HTML file
I save the Plotly figure as `sankey.html`.

In [ ]:
fig_sankey.write_html("sankey.html")

## Three Files Repo: hw5.ipynb, sankey.html, stacked_bar.html
https://elenazhuge.github.io/HW/

## Publicly Hosted Sankey Diagram
Live webpage link: https://elenazhuge.github.io/Elena-Zhuge_Data-Science-for-Public-Health-in-Python/